# Notebook 03 — Urban Heat และ Land Surface Temperature

**หัวข้อ:** Urban Heat และ Land Surface Temperature (LST)

**คำถามหลัก:** *อุณหภูมิพื้นผิวเมืองแตกต่างกันอย่างไร และสัมพันธ์กับลักษณะพื้นผิวอย่างไร?*

### วัตถุประสงค์การเรียนรู้
- แยก **Land Surface Temperature (LST)** ออกจาก air temperature
- เข้าใจ Urban Heat Island ในระดับแนวคิด
- เข้าใจบทบาทของ vegetation, impervious surface และ evapotranspiration
- ใช้ Landsat 8 Level-2 Surface Temperature
- เข้าใจ scale factor และหน่วย Kelvin/°C
- สร้าง LST map ที่มีช่วงสีและหน่วยเหมาะสม
- ใช้ “urban analysis zone” เพื่อหลีกเลี่ยงการสรุปจากทั้งจังหวัดอย่างไม่เหมาะสม

## 1. แนวคิดพื้นฐาน

$$
LST \neq T_{\mathrm{air}}
$$

LST คือ radiometric surface temperature ของพื้นผิวที่ satellite sensor ประเมิน ไม่ใช่อุณหภูมิอากาศมาตรฐานที่สถานีอุตุนิยมวิทยาวัดใกล้ระดับ 2 m

แนวคิด surface energy balance แบบง่าย:

$$
R_n = H + LE + G
$$

vegetation มักเพิ่ม evapotranspiration และ latent heat flux ขณะที่ impervious surface มักลด evapotranspiration และเพิ่ม heat storage

## การเตรียมระบบ (ทำทุก Notebook)

Notebook นี้ออกแบบสำหรับ **Google Colab + Google Earth Engine Python API + geemap**

### ก่อนเริ่ม
1. ต้องมีบัญชี Google Earth Engine ที่เปิดใช้งานแล้ว
2. ต้องมี Google Cloud Project ที่ใช้กับ Earth Engine
3. แก้ค่า `PROJECT` ใน code cell ด้านล่างให้เป็น Project ID ของตนเอง

> **แนวคิดสำคัญ:** Python ใน Colab เป็นฝั่งผู้ใช้ (client) ส่วน `ee.Image`, `ee.ImageCollection` และการคำนวณส่วนใหญ่เป็นวัตถุแบบ server-side ที่ประมวลผลบน Google Earth Engine

In [1]:
%pip install -q earthengine-api geemap pandas matplotlib scikit-learn scipy

import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Google Cloud Project ID ที่ลงทะเบียน Earth Engine แล้ว
PROJECT = "project-cf559859-d42a-44d4-925"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully.")
print("Using project:", PROJECT)

# -------------------------------------------------------------------
# ฟังก์ชันมาตรฐานสำหรับทุกแผนที่ในรายวิชา
# 1) สร้าง geemap.Map()
# 2) ลบ default OpenStreetMap ที่ถูก block ใน Colab environment นี้
# 3) ใช้ Esri.WorldImagery เป็น basemap เริ่มต้น
# 4) เตรียม Street Map และ Topographic Map ไว้ให้เลือกใน Layer Control
# -------------------------------------------------------------------
def create_course_map():
    m = geemap.Map()
    m.clear_layers()
    m.add_basemap("Esri.WorldImagery", show=True)
    m.add_basemap("Esri.WorldStreetMap", show=False)
    m.add_basemap("Esri.WorldTopoMap", show=False)
    return m

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 34.7 MB/s eta 0:00:00
Earth Engine initialized successfully.
Using project: project-cf559859-d42a-44d4-925


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


### Basemap ที่ใช้ในชุด Notebook นี้

ทุกแผนที่เรียกผ่าน `create_course_map()` เพื่อหลีกเลี่ยง default OpenStreetMap tile ที่ถูก block ใน Colab environment นี้ โดยกำหนด **Esri.WorldImagery** เป็นพื้นหลังเริ่มต้น และมี **Esri.WorldStreetMap** กับ **Esri.WorldTopoMap** ให้เปิดจาก Layer Control ตามวัตถุประสงค์ของการอ่านแผนที่


In [2]:
# Administrative boundary: FAO GAUL 2025 Level 1
admin1 = ee.FeatureCollection("FAO/GAUL/2025/level1")
thailand = admin1.filter(ee.Filter.eq("GAUL0_NAME", "Thailand"))

def get_admin1(name_contains):
    # เลือกหน่วยปกครองระดับ 1 ของประเทศไทยด้วยข้อความบางส่วนใน GAUL1_NAME
    return thailand.filter(
        ee.Filter.stringContains("GAUL1_NAME", name_contains)
    )

bangkok = get_admin1("Bangkok")
chiang_mai = get_admin1("Chiang Mai")

print("Bangkok features:", bangkok.size().getInfo())
print("Chiang Mai features:", chiang_mai.size().getInfo())

Bangkok features: 1
Chiang Mai features: 1


## 2. กำหนด Urban Analysis Zone

จาก Notebook 02 เราเห็นว่าการใช้ทั้งจังหวัดเชียงใหม่ทำให้พื้นที่ภูเขาและชนบทมีอิทธิพลต่อสถิติสูง

ใน Notebook นี้จึงกำหนด **วงกลมรัศมี 20 km รอบศูนย์กลางเมือง** เพื่อเป็น operational analysis zone สำหรับการเรียน

> ขอบเขตนี้ไม่ใช่ขอบเขตเมืองอย่างเป็นทางการ แต่เป็นพื้นที่วิเคราะห์ที่กำหนดอย่างชัดเจนและทำซ้ำได้

In [3]:
BKK_CENTER = ee.Geometry.Point([100.5018, 13.7563])
CM_CENTER  = ee.Geometry.Point([98.9853, 18.7883])

bkk_zone = BKK_CENTER.buffer(20_000)
cm_zone  = CM_CENTER.buffer(20_000)

Map_zone = create_course_map()
Map_zone.centerObject(cm_zone, 9)
Map_zone.addLayer(
    chiang_mai.style(color="8c8c8c", fillColor="00000000", width=1),
    {},
    "Chiang Mai province"
)
Map_zone.addLayer(
    ee.FeatureCollection([ee.Feature(cm_zone)]).style(
        color="202020", fillColor="00000000", width=3
    ),
    {},
    "20-km urban analysis zone"
)
Map_zone.add_layer_control()

Map_zone

Map(center=[18.788320517645563, 98.98530060100639], controls=(WidgetControl(options=['position', 'transparent_…

## 3. Landsat 8 Collection 2 Level 2

Dataset: `LANDSAT/LC08/C02/T1_L2`

`ST_B10` scale:
$$
T(K)=DN\times0.00341802+149.0
$$

แล้วแปลง Kelvin เป็น Celsius:
$$
T(^\circ C)=T(K)-273.15
$$

เรา filter เฉพาะ `PROCESSING_LEVEL = L2SP` เพื่อให้มี surface temperature

In [4]:
def prep_landsat8_l2(image):
    # QA_PIXEL bits 0-5: fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    qa = image.select("QA_PIXEL")
    qa_mask = qa.bitwiseAnd(0b111111).eq(0)
    saturation_mask = image.select("QA_RADSAT").eq(0)

    # Official Collection 2 scale factors
    optical = image.select("SR_B.").multiply(0.0000275).add(-0.2)
    lst_c = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )

    return (
        image.addBands(optical, None, True)
        .addBands(lst_c)
        .updateMask(qa_mask)
        .updateMask(saturation_mask)
        .copyProperties(image, ["system:time_start"])
    )

def landsat_hotseason(region, start="2024-02-01", end="2024-05-01"):
    ic = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
        .filter(ee.Filter.lt("CLOUD_COVER", 60))
        .map(prep_landsat8_l2)
    )
    composite = ic.median().clip(region)
    return ic, composite

ls_bkk, bkk_hot = landsat_hotseason(bkk_zone)
print("Landsat scenes:", ls_bkk.size().getInfo())

Landsat scenes: 10


## 4. สร้าง LST map — Bangkok

ใช้ช่วงสีเดียวกันสำหรับเมืองต่าง ๆ เมื่อมีวัตถุประสงค์เพื่อเปรียบเทียบเชิงปริมาณ เพื่อไม่ให้ auto-stretch ทำให้แผนที่สองเมืองดูร้อนเท่ากันทั้งที่ช่วงค่าต่างกัน

In [5]:
lst_vis = {
    "min": 20,
    "max": 45,
    "palette": ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
}

lst_bkk = bkk_hot.select("LST_C")

stats_bkk = lst_bkk.reduceRegion(
    reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            outputPrefix="",
            sharedInputs=True,
        ),
    geometry=bkk_zone,
    scale=30,
    maxPixels=1e9,
    tileScale=4,
)

print(stats_bkk.getInfo())

Map_lst = create_course_map()
Map_lst.centerObject(bkk_zone, 10)
Map_lst.addLayer(lst_bkk, lst_vis, "Bangkok LST")
Map_lst.addLayer(
    ee.FeatureCollection([ee.Feature(bkk_zone)]).style(
        color="202020", fillColor="00000000", width=2
    ),
    {},
    "20-km analysis zone"
)
Map_lst.add_colorbar(lst_vis, label="Land Surface Temperature (°C)", layer_name="Bangkok LST")
Map_lst.add_layer_control()
Map_lst

{'LST_C_max': 60.04367978000005, 'LST_C_mean': 42.37784505433785, 'LST_C_min': 28.397941610000032}


Map(center=[13.75632051764566, 100.50180058578509], controls=(WidgetControl(options=['position', 'transparent_…

### Observe the Map
ก่อนดูตัวเลข ให้ตอบ:
1. พื้นที่ LST สูงกระจุกตรงไหน?
2. แม่น้ำและพื้นที่สีเขียวปรากฏอย่างไร?
3. รูปแบบเป็น homogeneous หรือ heterogeneous?

## 5. แบบฝึกหัด — Chiang Mai

ทำ workflow เดียวกันสำหรับ `cm_zone`

**งาน**
- สร้าง hot-season Landsat composite
- แสดง LST ด้วย **ช่วงสี 20–45°C เดียวกับ Bangkok**
- คำนวณ mean/min/max
- เพิ่มขอบเขตจังหวัดเป็น contextual layer
- ตีความผลโดยพิจารณา built surface, vegetation และ topography

In [6]:
# TODO:
# ls_cm, cm_hot = landsat_hotseason(cm_zone)
# lst_cm = ...
# stats_cm = ...
# Map_cm_lst = ...

### คำถามการตีความ
1. จุดร้อนและจุดเย็นอยู่บริเวณใด?
2. mountain/vegetation ทางตะวันตกมีรูปแบบ LST อย่างไร?
3. ทำไมต้องใช้ color scale เดียวกันเมื่อต้องการเปรียบเทียบสองเมือง?
4. เหตุใด LST ช่วงหนึ่งจึงยังไม่เพียงพอที่จะสรุป “Urban Heat Island climatology”?

## 6. สิ่งที่ควรได้จาก Notebook 03

- LST เป็นตัวแปรด้านพื้นผิว ไม่ใช่ air temperature
- การกำหนด study area มีผลต่อสถิติ
- cartographic scale และ color scale มีผลต่อการตีความ
- หนึ่ง seasonal composite เป็น snapshot ไม่ใช่ climatology ระยะยาว

# SOLUTION — เปิดหลังทำแบบฝึกหัดแล้ว

In [7]:
ls_cm, cm_hot = landsat_hotseason(cm_zone)
lst_cm = cm_hot.select("LST_C")

stats_cm = lst_cm.reduceRegion(
    reducer=ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.minMax(),
            outputPrefix="",
            sharedInputs=True,
        ),
    geometry=cm_zone,
    scale=30,
    maxPixels=1e9,
    tileScale=4,
)

print("Landsat scenes:", ls_cm.size().getInfo())
print(stats_cm.getInfo())

Map_cm_lst = create_course_map()
Map_cm_lst.centerObject(cm_zone, 10)
Map_cm_lst.addLayer(lst_cm, lst_vis, "Chiang Mai LST")
Map_cm_lst.addLayer(
    chiang_mai.style(color="8c8c8c", fillColor="00000000", width=1),
    {},
    "Chiang Mai province"
)
Map_cm_lst.addLayer(
    ee.FeatureCollection([ee.Feature(cm_zone)]).style(
        color="202020", fillColor="00000000", width=2
    ),
    {},
    "20-km analysis zone"
)
Map_cm_lst.add_colorbar(lst_vis, label="Land Surface Temperature (°C)", layer_name="Chiang Mai LST")
Map_cm_lst.add_layer_control()
Map_cm_lst

Landsat scenes: 5
{'LST_C_max': 48.49077218000002, 'LST_C_mean': 34.7680147381881, 'LST_C_min': 23.289710720000016}


Map(center=[18.788320517645563, 98.98530060100639], controls=(WidgetControl(options=['position', 'transparent_…

### แนวคำตอบ
เชียงใหม่มี strong topographic control มากกว่ากรุงเทพฯ การวิเคราะห์ LST จึงต้องระวังการอธิบายทุกความแตกต่างว่าเกิดจาก urbanization เพียงอย่างเดียว เพราะ elevation, vegetation และ terrain มีบทบาทร่วมด้วย

## แหล่งข้อมูลอ้างอิง
- Landsat 8 Level 2 Collection 2 Tier 1: https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2